In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df_books=pd.read_csv('/kaggle/input/datasets/arashnic/book-recommendation-dataset/Books.csv')
df_books.head()

In [ ]:
df_users=pd.read_csv('/kaggle/input/datasets/arashnic/book-recommendation-dataset/Users.csv')
df_users.head()

In [ ]:
df_ratings=pd.read_csv('/kaggle/input/datasets/arashnic/book-recommendation-dataset/Ratings.csv')
df_ratings.head()

In [ ]:
df_books.info()

In [ ]:
len(df_books['Book-Title'].unique())

In [ ]:
df_users.info()
len(df_users['Location'].unique())

In [ ]:
df_ratings.info()
len(df_ratings['ISBN'].unique())

In [ ]:
print(df_books.duplicated().sum())
print(df_users.duplicated().sum())
print(df_ratings.duplicated().sum())

# **using popularity based Recommender System **

In [ ]:
df_ratings=df_ratings.merge(df_books,on='ISBN')

In [ ]:
df_ratings

In [ ]:
num_df_ratings=df_ratings.groupby('Book-Title').count()[['Book-Rating']].reset_index()
num_df_ratings.rename(columns={'Book-Rating':'Number of Ratings'},inplace=True)

In [ ]:
df_ratings.info()

In [ ]:
num_df_ratings=num_df_ratings.merge(df_ratings[['Book-Title','Book-Rating']].groupby('Book-Title').mean().reset_index(),on='Book-Title')

In [ ]:
num_df_ratings.rename(columns={'Book-Rating':'Avg-Rating'},inplace=True)

In [ ]:
num_df_ratings['Number of Ratings'].describe()

In [ ]:
num_df_ratings=num_df_ratings[num_df_ratings['Number of Ratings']>250].sort_values('Avg-Rating',ascending=False).head(50)

In [ ]:
num_df_ratings_merged=num_df_ratings.merge(df_books,on='Book-Title')

In [ ]:
num_df_ratings_merged.drop_duplicates('Book-Title',inplace=True)
num_df_ratings

# Using Collaborative Filtering System

In [ ]:
df_greater_200=df_ratings.groupby('User-ID').count()
indexes=df_greater_200[df_greater_200['Book-Rating']>200].index

In [ ]:
y=df_ratings[df_ratings['User-ID'].isin(indexes)]
df_greater_50=y.groupby('Book-Title').count()
books_fin=df_greater_50[df_greater_50['Book-Rating']>50].index


In [ ]:
books_fin

In [ ]:
final_ratings=y[y['Book-Title'].isin(books_fin)]

In [ ]:
final_ratings.head()

In [ ]:
pivot_table=final_ratings.pivot_table(index='Book-Title',columns='User-ID',values='Book-Rating')

In [ ]:
pivot_table

In [ ]:
pivot_table.fillna(0,inplace=True)

In [ ]:
pivot_table

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
cs=cosine_similarity(pivot_table)

In [ ]:
def recommend_books(book_name):
    index=np.where(pivot_table.index==book_name)[0][0]
    distances=cs[index]
    top_distances=sorted(list(enumerate(distances)),key=lambda x:-x[1])[1:11]
    for index,_ in top_distances:
        print(pivot_table.index[index])

In [ ]:
recommend_books('1984')